# 10장. 분류 분석으로 주문 취소 여부 예측하기

## 제출 정보

이름: 박성주 / GitHub ID: mikeypark / 작성일: 2026-11-13
최종 Notebook URL: https://github.com/mikeypark/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 학습 목표

accuracy 하나만 보고 모델을 선택하는 게 아니라 클래스 비율, baseline, precision/recall/F1, threshold, FP/FN의 업무적 의미를 함께 판단하는 흐름을 익히는 게 목표다. 모델이 baseline보다 나아졌는지, 나아졌다면 어느 지표에서 의미 있게 나아졌는지를 직접 확인하고 기록하는 것에 집중했다.

## 1. 분류 문제 정의

- 타깃: `is_cancelled` (`order_status == 'cancelled'` -> 1, 아니면 0)
- 0의 의미: 주문이 완료(completed) 또는 환불(refunded)된 경우
- 1의 의미: 주문이 취소(cancelled)된 경우
- 예측 시점: 주문이 접수된 직후, 취소 여부가 확정되기 전
- 예측 시점에 사용할 수 있는 정보: `payment_method`, `order_month`, `order_dayofweek`, `gender`, `age`, `city`
- 예측 이후에만 알 수 있어 제외한 정보: `order_status`(타깃 자체), `line_total`, `quantity`, `unit_price`(주문 처리 후에 확정되는 값)

### 나의 해석과 판단

`refunded`를 1에 포함하지 않은 이유는 환불은 주문이 한 번 완료된 뒤 고객이 반품 절차를 밟는 것이고, 취소는 주문이 처리되기 전에 취소하는 것이라서 발생 메커니즘이 다르다. 두 경우를 같은 타깃으로 묶으면 모델이 학습할 패턴이 섞여서 더 혼란스러울 수 있다고 판단했다. 취소 여부 예측이라는 목적에 충실하게 `cancelled`만 1로 정의했다.

## 2. 패키지와 경로 설정

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

CURRENT_DIR  = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'chapter10' else CURRENT_DIR
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
IMAGE_DIR = (
    CURRENT_DIR / 'images' if CURRENT_DIR.name == 'chapter10'
    else PROJECT_ROOT / 'chapter10' / 'images'
)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print('프로젝트 루트:', PROJECT_ROOT)
print('이미지 폴더:', IMAGE_DIR)

## 3. 전처리 데이터 불러오기

In [ ]:
customers   = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders      = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')

orders['order_date']     = pd.to_datetime(orders['order_date'], errors='coerce')
orders['order_month']    = orders['order_date'].dt.to_period('M').astype(str)
orders['order_dayofweek']= orders['order_date'].dt.dayofweek

# 타깃 생성
orders['is_cancelled'] = (orders['order_status'] == 'cancelled').astype(int)

print('customers:', customers.shape)
print('orders:   ', orders.shape)
print('is_cancelled 분포:')
print(orders['is_cancelled'].value_counts())

## 4. 모델링 데이터 구성 (leakage 차단)

In [ ]:
model_data = (
    orders
    .merge(customers[['customer_id', 'gender', 'age', 'city']], on='customer_id', how='left')
    .sort_values('order_date')
    .reset_index(drop=True)
)

NUMERIC_FEATURES     = ['order_dayofweek', 'age']
CATEGORICAL_FEATURES = ['payment_method', 'order_month', 'gender', 'city']
FEATURE_COLUMNS      = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET               = 'is_cancelled'

# leakage 체크
FORBIDDEN = ['order_status', 'is_cancelled', 'line_total', 'quantity', 'unit_price']
leak_check = pd.DataFrame({
    'feature': FEATURE_COLUMNS + FORBIDDEN,
    '예측시점_사용가능': ['Y']*len(FEATURE_COLUMNS) + ['N']*len(FORBIDDEN),
    '최종_사용': ['Y']*len(FEATURE_COLUMNS) + ['N']*len(FORBIDDEN),
})
print('모델링 데이터:', model_data.shape)
print('사용 feature:', FEATURE_COLUMNS)
leak_check

### 결과 관찰

`order_status`는 타깃 자체이므로 feature에서 제외했다. `line_total`, `quantity`, `unit_price`는 주문이 처리된 뒤에야 확정되는 값이라서 예측 시점에 알 수 없다. 최종적으로 쓰는 feature는 `order_dayofweek`, `age`, `payment_method`, `order_month`, `gender`, `city` 여섯 개다.

## 5. 클래스 비율 확인

In [ ]:
class_counts = orders['is_cancelled'].value_counts().sort_index()
class_ratio  = orders['is_cancelled'].value_counts(normalize=True).sort_index()

class_df = pd.DataFrame({
    '클래스': ['0 (취소 아님)', '1 (취소)'],
    '개수': class_counts.values,
    '비율(%)': (class_ratio.values * 100).round(1),
})
print(class_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['steelblue', 'coral']
bars = ax.bar(class_df['클래스'], class_df['개수'], color=colors, edgecolor='white', width=0.5)
for bar, pct, cnt in zip(bars, class_df['비율(%)'], class_df['개수']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{int(cnt)}건 ({pct}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('클래스 비율 (is_cancelled)', fontsize=12)
ax.set_ylabel('주문 수', fontsize=10)
ax.set_ylim(0, 280)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'step02_class_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('step02_class_ratio.png 저장 완료')

## 2. 클래스 비율

| 클래스 | 개수 | 비율 |
| --- | ---: | ---: |
| 0 (취소 아님) | 236 | 78.7% |
| 1 (취소) | 64 | 21.3% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰

전체 300건 중 취소는 64건(21.3%)이다. 취소 아님이 236건(78.7%)으로 훨씬 많다.

### 나의 해석과 판단

78.7%가 취소가 아니기 때문에 모델이 아무것도 하지 않고 전부 0으로 예측해도 accuracy가 78.7%가 나온다. 이 상황에서 accuracy 단일 지표만 보면 성능이 좋아 보여도 실제로는 취소 주문을 하나도 잡지 못하는 모델일 수 있다. 그래서 recall(취소 주문을 얼마나 잡아냈는지)과 F1을 주요 지표로 함께 봐야 한다.

## 6. 데이터 분할 (시간 순서)

In [ ]:
n = len(model_data)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

train_data = model_data.iloc[:train_end].copy()
val_data   = model_data.iloc[train_end:val_end].copy()
test_data  = model_data.iloc[val_end:].copy()

X_train, y_train = train_data[FEATURE_COLUMNS], train_data[TARGET]
X_val,   y_val   = val_data[FEATURE_COLUMNS],   val_data[TARGET]
X_test,  y_test  = test_data[FEATURE_COLUMNS],  test_data[TARGET]

split_summary = pd.DataFrame([
    {'split': 'train',      'start': train_data['order_date'].min().date(), 'end': train_data['order_date'].max().date(), 'n': len(train_data), 'cancelled': int(y_train.sum())},
    {'split': 'validation', 'start': val_data['order_date'].min().date(),   'end': val_data['order_date'].max().date(),   'n': len(val_data),   'cancelled': int(y_val.sum())},
    {'split': 'final_test', 'start': test_data['order_date'].min().date(),  'end': test_data['order_date'].max().date(),  'n': len(test_data),  'cancelled': int(y_test.sum())},
])
split_summary

### 결과 관찰

주문 날짜 순으로 정렬한 뒤 앞 80%를 train(240건), 다음 10%를 validation(30건), 마지막 10%를 final test(30건)로 나눴다. val에 취소 주문이 3건밖에 없어서 validation 지표가 불안정할 수 있다는 점을 감안하고 해석해야 한다.

## 7. 전처리 파이프라인 정의

In [ ]:
num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, NUMERIC_FEATURES),
    ('cat', cat_pipe, CATEGORICAL_FEATURES),
])
print('숫자형 feature:', NUMERIC_FEATURES)
print('범주형 feature:', CATEGORICAL_FEATURES)

## 8. Dummy Baseline 및 모델 학습

In [ ]:
def evaluate_clf(name, y_true, y_pred):
    return {
        'model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 4),
    }

# Dummy (most_frequent -> 항상 0 예측)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_dummy_val = dummy.predict(X_val)

# LogisticRegression
lr_pipe = Pipeline([('pre', preprocessor), ('m', LogisticRegression(max_iter=1000, random_state=42))])
lr_pipe.fit(X_train, y_train)
y_lr_prob_val = lr_pipe.predict_proba(X_val)[:, 1]
y_lr_val_50   = lr_pipe.predict(X_val)                          # 기본 thresh=0.5
y_lr_val_20   = (y_lr_prob_val >= 0.20).astype(int)             # thresh=0.20

comp = pd.DataFrame([
    evaluate_clf('Dummy (most_frequent)',          y_val, y_dummy_val),
    evaluate_clf('LogisticRegression (thresh=0.5)', y_val, y_lr_val_50),
    evaluate_clf('LogisticRegression (thresh=0.20)', y_val, y_lr_val_20),
])
print(comp.to_string(index=False))

# 그래프
metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
dummy_scores = [comp.loc[0, m] for m in metrics]
lr_scores    = [comp.loc[2, m] for m in metrics]
x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, dummy_scores, w, label='Dummy (most_frequent)', color='gray', edgecolor='white')
b2 = ax.bar(x + w/2, lr_scores,    w, label='LogisticRegression (thresh=0.20)', color='steelblue', edgecolor='white')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.set_title('Dummy vs LogisticRegression (Validation)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'step03_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('step03_baseline.png 저장 완료')

## 3. Dummy Baseline

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy | 0.9000 | 0.0000 | 0.0000 | 0.0000 |
| LogisticRegression (thresh=0.20) | 0.7667 | 0.3000 | 1.0000 | 0.4615 |

![Baseline 비교](images/step03_baseline.png)

### 결과 관찰

Dummy는 모든 주문을 0(취소 아님)으로 예측해서 accuracy가 90%지만 recall은 0이다. 취소 주문 3건을 하나도 잡지 못한다는 뜻이다. LogisticRegression은 threshold를 0.20으로 낮추면 recall이 1.0으로 올라간다. 취소 3건을 모두 찾아냈지만 FP가 7건 발생해서 precision은 0.30이다.

### 나의 해석과 판단

threshold=0.5(기본값)에서 LR도 dummy와 똑같이 취소를 하나도 잡지 못한다. 클래스 불균형이 심해서 모델이 취소 클래스를 예측하려면 threshold를 낮춰야 한다. threshold=0.20에서는 recall이 1.0으로 baseline보다 의미 있게 나아졌다. accuracy는 76.7%로 baseline(90%)보다 낮아졌는데, 이건 FP가 7건 늘어난 결과다. 취소 감지가 목적이라면 accuracy 하락을 감수하고 recall을 올리는 쪽이 맞는 방향이다.

### 업무·분석적 의미

주문 취소를 미리 감지해서 예방 조치(할인 쿠폰 발송, 고객 연락 등)를 취하는 시나리오라면 FN(취소인데 놓침)이 FP(취소 아닌데 취소로 예측)보다 더 부담스럽다. 취소를 놓치면 아무 조치도 없이 취소가 발생하고, 취소 아닌 주문에 불필요한 쿠폰을 보내는 건 비용은 있어도 취소를 막지는 못한다. 그래서 recall을 더 중요하게 보는 쪽이 타당하다.

### 한계와 추가 확인 사항

val에 취소 주문이 3건뿐이라 recall=1.0이 의미 있는 수치인지 판단하기 어렵다. 취소 주문이 더 많은 기간 데이터로 검증해야 신뢰할 수 있다.

## 4. Validation 성능과 우선 지표

- Accuracy: 0.7667
- Precision: 0.3000
- Recall: 1.0000
- F1: 0.4615

### 내가 가장 중요하게 본 지표

Recall

### 그 이유

주문 취소를 사전에 감지해서 예방 조치를 취하는 시나리오에서 취소를 놓치는 FN이 더 부담스럽다. FP(취소 아닌데 취소로 예측)는 불필요한 쿠폰 발송 등의 비용이 발생하지만, FN(취소인데 놓침)은 취소 자체를 막을 기회를 완전히 잃는다. 단, 실제 FP/FN 비용 정보가 없어서 이 판단은 가정에 기반한다.

## 9. Threshold 비교

In [ ]:
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
thresh_rows = []
for t in thresholds:
    yt = (y_lr_prob_val >= t).astype(int)
    thresh_rows.append({
        'Threshold': t,
        'Precision': round(precision_score(y_val, yt, zero_division=0), 3),
        'Recall':    round(recall_score(y_val, yt, zero_division=0), 3),
        'F1':        round(f1_score(y_val, yt, zero_division=0), 3),
        'pred_pos':  int(yt.sum()),
    })
thresh_df = pd.DataFrame(thresh_rows)
print(thresh_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, thresh_df['Precision'], 'o-', label='Precision', color='coral')
ax.plot(thresholds, thresh_df['Recall'],    's-', label='Recall',    color='steelblue')
ax.plot(thresholds, thresh_df['F1'],        '^-', label='F1',        color='green')
ax.axvline(0.20, color='gray', linestyle='--', linewidth=1, label='선택 Threshold (0.20)')
ax.set_xlabel('Threshold', fontsize=10)
ax.set_ylabel('Score', fontsize=10)
ax.set_ylim(-0.05, 1.15)
ax.set_title('Threshold별 Precision / Recall / F1 (Validation, LogisticRegression)', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'step05_threshold.png', dpi=150, bbox_inches='tight')
plt.show()
print('step05_threshold.png 저장 완료')

## 5. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.15 | 0.136 | 1.000 | 0.240 | FP 많음, 취소 전부 감지 |
| 0.20 | 0.300 | 1.000 | 0.462 | F1 최고, 취소 전부 감지 |
| 0.25 | 0.333 | 0.333 | 0.333 | FP 감소, recall 급락 |
| 0.30 | 0.333 | 0.333 | 0.333 | 0.25와 동일 |
| 0.50 | 0.000 | 0.000 | 0.000 | 취소 예측 없음 |

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold

0.20

### 선택 이유

0.20이 F1 기준으로 가장 높고(0.462), recall도 1.0으로 취소 주문을 전부 감지한다. 0.15는 pred_pos가 22건으로 너무 많아서 FP 비용이 크고, 0.25부터는 recall이 0.333으로 급락해서 취소의 2/3를 놓친다.

### 나의 해석과 판단

threshold는 final test를 보지 않고 validation에서 선택해야 한다. final test를 보면서 threshold를 조정하면 final test에 과적합되어 실제 운영 성능을 과대평가하게 된다. validation에서 F1과 recall 기준으로 0.20을 선택하고, final test는 그 선택이 일반화됐는지 확인하는 1회 검증 용도로만 사용했다.

## 10. FP/FN 해석

In [ ]:
cm = confusion_matrix(y_val, y_lr_val_20)
print('Confusion Matrix (validation, thresh=0.20):')
print(cm)
print(f'TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['예측 0 (취소 아님)', '예측 1 (취소)'],
            yticklabels=['실제 0 (취소 아님)', '실제 1 (취소)'])
ax.set_title('Confusion Matrix - Validation (thresh=0.20)', fontsize=11)
ax.set_xlabel('예측값', fontsize=10)
ax.set_ylabel('실제값', fontsize=10)
plt.tight_layout()
plt.savefig(IMAGE_DIR / 'step06_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('step06_confusion_matrix.png 저장 완료')

## 6. FP/FN 해석

- FP 의미: 실제로는 취소하지 않을 주문인데 취소할 것이라고 예측한 경우 (27건 중 7건)
- FN 의미: 실제로 취소할 주문인데 취소하지 않을 것이라고 예측한 경우 (3건 중 0건)
- 현재 목적에서 더 부담스러울 수 있는 오류: FN (취소를 놓침)
- 그 판단에 추가로 필요한 업무 정보: 취소 1건을 놓쳤을 때의 손실(매출 손실, 재고 처리 비용 등)과, FP 1건에 발생하는 비용(쿠폰 발송 비용, 고객 경험 저하 등)의 실제 금액

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미

val에서 FN=0이다. 취소 3건을 모두 감지했다. 반면 FP가 7건으로 전체 val 주문 30건의 23%에 해당한다. FP 주문에 불필요한 예방 쿠폰을 보내면 쿠폰 비용이 발생하고 고객이 오히려 불필요한 혜택을 받는 상황이 생긴다. 취소 예방 비용보다 쿠폰 비용이 훨씬 작다면 감수할 수 있지만, 그 판단은 실제 단가 정보가 있어야 한다.

### 한계와 추가 확인 사항

val에 취소가 3건뿐이라 FN=0이라는 수치가 운이 좋은 결과일 수 있다. 더 많은 취소 주문이 포함된 기간에서 검증해야 FN=0이 일반적인지 확인할 수 있다.

## 11. Final Test 평가

In [ ]:
y_lr_prob_test = lr_pipe.predict_proba(X_test)[:, 1]
y_lr_test_20   = (y_lr_prob_test >= 0.20).astype(int)

final_rows = [
    evaluate_clf('Dummy (most_frequent)',           y_test, dummy.predict(X_test)),
    evaluate_clf('LogisticRegression (thresh=0.20)', y_test, y_lr_test_20),
]
final_comp = pd.DataFrame(final_rows)
print('=== Validation ===')
print(comp.to_string(index=False))
print()
print('=== Final Test ===')
print(final_comp.to_string(index=False))
print()
cm_test = confusion_matrix(y_test, y_lr_test_20)
print('Confusion Matrix (final test, thresh=0.20):')
print(cm_test)
print(f'TN={cm_test[0,0]}, FP={cm_test[0,1]}, FN={cm_test[1,0]}, TP={cm_test[1,1]}')

## 7. Final Test

- Accuracy: 0.6000
- Precision: 0.3333
- Recall: 0.5000
- F1: 0.4000
- Validation 대비 변화: Accuracy 0.7667 -> 0.6000, Recall 1.0000 -> 0.5000, F1 0.4615 -> 0.4000

### 결과 관찰

final test에서 취소 주문 8건 중 4건을 감지했다(recall=0.50). validation에서 recall이 1.0이었던 것과 비교하면 크게 떨어졌다. TN=14, FP=8, FN=4, TP=4다. FP 8건, FN 4건이 발생했다. accuracy는 0.60으로 val(0.767)보다 낮아졌다.

### 나의 해석과 판단

val recall=1.0이 운이 좋은 결과였다는 게 final test에서 확인됐다. val에 취소가 3건뿐이었고, 그 3건이 마침 모두 모델 확률이 0.20 이상인 주문들이었을 뿐이다. final test에서 recall이 0.50으로 낮아진 건 모델이 완전히 일반화됐다고 볼 수 없다는 뜻이다. 그래도 dummy(recall=0)보다는 나아진 건 맞고, F1도 dummy(0)보다 높다. 모델이 baseline보다 낫긴 하지만 아직 운영에 쓰기에는 불안정하다.

## 12. 실습 과제

### 과제 1. `RandomForestClassifier`로 같은 분석을 반복하고 LogisticRegression과 F1을 비교하세요.

In [ ]:
# 과제 1. RandomForest vs LogisticRegression
rf_pipe = Pipeline([('pre', preprocessor), ('m', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_pipe.fit(X_train, y_train)

y_rf_prob_val = rf_pipe.predict_proba(X_val)[:, 1]
# RF에서 최적 threshold 탐색
rf_thresh_rows = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.50]:
    yt = (y_rf_prob_val >= t).astype(int)
    rf_thresh_rows.append({
        'Threshold': t,
        'Precision': round(precision_score(y_val, yt, zero_division=0), 3),
        'Recall':    round(recall_score(y_val, yt, zero_division=0), 3),
        'F1':        round(f1_score(y_val, yt, zero_division=0), 3),
    })
rf_thresh_df = pd.DataFrame(rf_thresh_rows)
print('RF Threshold 탐색:')
print(rf_thresh_df.to_string(index=False))

# RF 최적 threshold=0.25로 비교
y_rf_val_best = (y_rf_prob_val >= 0.25).astype(int)
compare_df = pd.DataFrame([
    evaluate_clf('LR (thresh=0.20)',  y_val, y_lr_val_20),
    evaluate_clf('RF (thresh=0.25)',  y_val, y_rf_val_best),
])
print()
print('모델 비교 (validation):')
print(compare_df.to_string(index=False))

### 결과 관찰

RandomForest는 threshold=0.25에서 P=0.143, R=0.667, F1=0.235다. threshold=0.30에서는 P=0.154, R=0.667, F1=0.250이다. LR(thresh=0.20)의 F1=0.462에 비해 RF는 최선의 threshold에서도 F1이 0.25 수준에 머문다. LR이 이 데이터에서 더 나은 선택이다. RF의 probability 범위가 0.07~0.49로 좁아서, threshold를 조정해도 취소 주문을 충분히 높은 확률로 예측하지 못한다는 게 원인이다.

### 과제 2. 클래스 불균형을 처리하기 위해 `class_weight='balanced'`를 사용했을 때 성능이 어떻게 달라지는지 확인하세요.

In [ ]:
# 과제 2. class_weight='balanced'
lr_balanced = Pipeline([
    ('pre', preprocessor),
    ('m', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
lr_balanced.fit(X_train, y_train)
y_balanced_val = lr_balanced.predict(X_val)
y_balanced_prob = lr_balanced.predict_proba(X_val)[:, 1]

balanced_rows = []
for t in [0.30, 0.40, 0.50, 0.60]:
    yt = (y_balanced_prob >= t).astype(int)
    balanced_rows.append({
        'Threshold': t,
        'Precision': round(precision_score(y_val, yt, zero_division=0), 3),
        'Recall':    round(recall_score(y_val, yt, zero_division=0), 3),
        'F1':        round(f1_score(y_val, yt, zero_division=0), 3),
        'pred_pos':  int(yt.sum()),
    })
print('LR balanced threshold 탐색:')
print(pd.DataFrame(balanced_rows).to_string(index=False))

comp2 = pd.DataFrame([
    evaluate_clf('LR (thresh=0.20)',           y_val, y_lr_val_20),
    evaluate_clf('LR balanced (thresh=0.5)',   y_val, y_balanced_val),
])
print()
print(comp2.to_string(index=False))

### 결과 관찰

`class_weight='balanced'`를 쓰면 LR이 기본 threshold=0.5에서도 소수 클래스를 예측하려는 경향이 강해진다. balanced LR의 probability 범위가 더 넓어져서 threshold 조정 없이도 recall이 올라간다. 단 FP도 같이 늘어나서 precision은 낮아진다. 클래스 불균형을 다루는 방법이 threshold 조정 외에도 가중치 조정이라는 접근이 있다는 걸 확인했다.

### 과제 3. `order_status`의 `refunded`도 1에 포함했을 때 클래스 비율과 모델 성능이 어떻게 달라지는지 비교하세요.

In [ ]:
# 과제 3. refunded 포함 시 비교
model_data2 = model_data.copy()
model_data2['is_not_completed'] = (model_data2['order_status'].isin(['cancelled','refunded'])).astype(int)

print('타깃 비교:')
print(pd.DataFrame({
    '타깃': ['is_cancelled (취소만)', 'is_not_completed (취소+환불)'],
    '0 (정상)': [236, 184],
    '1 (비정상)': [64, 116],
    '1 비율(%)': [21.3, 38.7],
}).to_string(index=False))

TARGET2 = 'is_not_completed'
train2 = model_data2.iloc[:train_end]
val2   = model_data2.iloc[train_end:val_end]
X_tr2, y_tr2 = train2[FEATURE_COLUMNS], train2[TARGET2]
X_va2, y_va2 = val2[FEATURE_COLUMNS],   val2[TARGET2]

lr2 = Pipeline([('pre', preprocessor), ('m', LogisticRegression(max_iter=1000, random_state=42))])
lr2.fit(X_tr2, y_tr2)
y_lr2_prob = lr2.predict_proba(X_va2)[:, 1]
y_lr2_20   = (y_lr2_prob >= 0.20).astype(int)

comp3 = pd.DataFrame([
    evaluate_clf('취소만 (is_cancelled)',         y_val,  y_lr_val_20),
    evaluate_clf('취소+환불 (is_not_completed)',  y_va2,  y_lr2_20),
])
print()
print('모델 성능 비교 (validation, thresh=0.20):')
print(comp3.to_string(index=False))

### 결과 관찰

환불까지 포함하면 1 클래스 비율이 21.3%에서 38.7%로 올라가서 클래스 불균형이 완화된다. val에서 1 클래스 주문 수도 늘어나서 평가가 더 안정적이 된다. 단 타깃 정의가 달라지기 때문에 두 모델을 직접 비교하는 건 의미가 없고, 취소와 환불을 같은 "비정상 주문"으로 처리할 업무 근거가 있는지를 먼저 판단해야 한다.

### 과제 4. LLM이 제안한 분류 코드에서 leakage가 될 수 있는 feature를 검토하세요.

In [ ]:
# 과제 4. LLM 제안 코드 leakage 검토
llm_review = pd.DataFrame({
    'LLM_제안_feature': [
        'payment_method', 'order_month', 'order_dayofweek', 'gender', 'age', 'city',
        'order_status',   # leakage: 타깃 자체
        'line_total',     # leakage: 주문 처리 후 확정
        'quantity',       # leakage
        'unit_price',     # leakage
    ],
    'leakage_여부': [
        'N', 'N', 'N', 'N', 'N', 'N',
        'Y - 타깃 자체 (order_status에서 is_cancelled 생성)',
        'Y - 주문 처리 후 확정되는 값',
        'Y - 주문 처리 후 확정되는 값',
        'Y - 주문 처리 후 확정되는 값',
    ],
    '조치': ['사용']*6 + ['제거']*4,
})
llm_review

### 결과 관찰

LLM이 제안한 feature에 `order_status`가 직접 포함된 경우가 있었다. 타깃인 `is_cancelled`가 `order_status`에서 파생된 값이기 때문에, `order_status`를 feature로 넣으면 타깃 정보가 그대로 입력에 들어가는 완벽한 leakage다. `line_total`, `quantity`, `unit_price`도 주문이 처리된 뒤에야 확정되는 값이라서 예측 시점에 알 수 없다. LLM이 예측 시점에 대한 명시적인 제약을 모른 채 모든 컬럼을 나열한 결과다. 타깃 생성 방식과 예측 시점을 LLM 프롬프트에 구체적으로 넣어줬을 때 leakage가 줄어드는지 확인하는 작업이 필요하다.

### 과제 5. 취소율이 높은 고객 특성을 탐색하고 그 특성이 feature 중요도와 일치하는지 비교하세요.

In [ ]:
# 과제 5. 취소율 높은 고객 특성
train_full = model_data.iloc[:train_end].copy()

print('결제수단별 취소율:')
pm_cancel = train_full.groupby('payment_method')['is_cancelled'].agg(['mean','sum','count'])
pm_cancel.columns = ['취소율', '취소건수', '총건수']
pm_cancel['취소율(%)'] = (pm_cancel['취소율'] * 100).round(1)
print(pm_cancel[['취소율(%)','취소건수','총건수']].sort_values('취소율(%)', ascending=False))

print('\n도시별 취소율:')
city_cancel = train_full.groupby('city')['is_cancelled'].agg(['mean','sum','count'])
city_cancel.columns = ['취소율', '취소건수', '총건수']
city_cancel['취소율(%)'] = (city_cancel['취소율'] * 100).round(1)
print(city_cancel[['취소율(%)','취소건수','총건수']].sort_values('취소율(%)', ascending=False))

# LR 계수 (feature 중요도 대용)
print('\nLR 계수 절대값 상위 10개 (preprocessor 통과 후):')
feature_names = (lr_pipe.named_steps['pre']
                 .get_feature_names_out())
coefs = pd.Series(np.abs(lr_pipe.named_steps['m'].coef_[0]), index=feature_names)
print(coefs.sort_values(ascending=False).head(10))

### 결과 관찰

도시별·결제수단별 취소율 차이가 있는지 확인했다. LR 계수 절대값이 큰 feature와 취소율이 높은 그룹이 일치하면 모델이 실제 패턴을 학습했다는 간접 증거가 된다. 특정 도시나 결제수단에서 취소율이 유독 높다면, 그 그룹에 대한 사전 개입이 취소 예방에 더 효과적일 수 있다. 단 샘플 수가 작아서 비율의 신뢰도가 낮다는 점은 감안해야 한다.

## 13. 스크립트 재실행 증거

In [ ]:
import subprocess
result = subprocess.run(
    ['python', str(PROJECT_ROOT / 'scripts' / 'run_classification_analysis.py')],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr)

## 14. LLM 활용 기록

### 사용 목적

두 가지 상황에서 LLM을 활용했다. 첫 번째는 과제 4를 위해 LLM에게 분류 모델 코드를 요청해서 어떤 feature를 제안하는지 직접 확인하는 용도였다. 두 번째는 threshold를 낮춰서 recall을 올리는 접근이 맞는지 확인하는 용도였다.

### 결과 관찰

첫 번째 요청에서 LLM이 `order_status`, `line_total`, `quantity`를 feature 목록에 포함했다. 예측 시점을 명시하지 않은 프롬프트에서는 타깃 생성 소스와 처리 후 확정 값을 구분하지 못한다는 걸 확인했다. 두 번째 요청에서는 threshold를 낮추면 recall이 올라가지만 precision이 낮아지는 trade-off를 설명받았는데, 이미 코드로 확인한 내용과 일치했다. LLM은 개념 확인에는 유용하지만, feature leakage 여부는 항상 직접 확인해야 한다.

## 8. 최종 사용 판단

- [ ] 참고용 사용 가능
- [x] 추가 검증 후 사용 가능
- [ ] 현재 사용 보류

### 판단 근거

1. dummy(recall=0, F1=0)보다 나은 성능이다. final test에서도 recall=0.50, F1=0.40으로 baseline보다 의미 있게 나아졌다.
2. val recall=1.0이 final test에서 0.50으로 절반으로 떨어졌다. val 샘플(취소 3건)이 너무 작아서 val 지표가 과적합됐을 가능성이 있고, 더 많은 기간 데이터로 검증이 필요하다.
3. final test FP=8, FN=4다. FP가 FN의 2배다. recall을 높이려면 FP를 감수해야 하는 trade-off가 있고, 실제 FP/FN 비용 비율이 확정돼야 threshold 선택을 최종 확정할 수 있다.

### 현재 모델의 가장 큰 위험

val에 취소 주문이 3건뿐이어서 threshold 선택 근거가 너무 적은 샘플에 기반한다. threshold=0.20이 다른 기간에서도 유효한지 아직 검증되지 않았다.

### 다음 개선 우선순위

1. 더 긴 기간의 데이터를 확보해서 val 취소 샘플을 늘리고 threshold를 재검토한다.
2. `class_weight='balanced'` 또는 오버샘플링(SMOTE)을 적용해서 클래스 불균형을 처리한 성능과 비교한다.
3. 실제 FP/FN 비용 정보를 확인해서 precision과 recall 중 무엇을 우선할지 근거를 확정한다.

## 최종 체크

- [x] 타깃과 예측 시점을 정의했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] precision/recall/F1을 해석했습니다.
- [x] Threshold 선택 이유를 작성했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Validation과 Final Test를 구분했습니다.
- [x] 최종 Notebook URL을 제출합니다.